# Independent local Bell settings: raw blocks

This notebook runs the existing `two_qutrit`, `ghz3`, and `ame43` Bell functionals with independently sampled local settings. It creates portable reference-state QPY inputs in two encodings, executes local Aer comparisons, and shows separately guarded IBM, IQM, and PIAST/AQT examples through the same public API.

Each draw selects one setting per party, independently and uniformly, then measures a joint result for `shots_per_draw` shots. Repeated settings remain separate blocks. Every comparison arm creates its own schedule. Production settings use Python `secrets`; this is system randomness, not a physical QRNG. The simulator seed only controls simulated measurement outcomes.

The small budgets below demonstrate the workflow; they are not a hardware-budget recommendation.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import numpy as np
from qiskit import QuantumCircuit, qpy
from qiskit.circuit.library import DiagonalGate, StatePreparation

from qudits_on_qubits.experiments import (
    AerIdeal, IBMHardware, IQMHardware, PiastQHardware,
    ExperimentSpec, PathBasis, RandomizedBlocks,
    run_experiment, resume_experiment,
)
from qudits_on_qubits.reference_experiments import get_reference_experiment

REPO_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()
     and (directory / "src" / "qudits_on_qubits").is_dir()),
    Path.cwd(),
)
WORK_ROOT = REPO_ROOT / "artifacts" / "bell_randomized_raw"
STATES = ("two_qutrit", "ghz3", "ame43")
ENCODINGS = {
    "canonical": np.eye(4, dtype=complex)[:, :3],
    "permuted": np.eye(4, dtype=complex)[:, [2, 0, 3]],
}
MEASUREMENT = RandomizedBlocks(
    setting_draws=64,
    shots_per_draw=16,
    max_setting_draws=512,
    max_circuits_per_job=100,
    confidence_level=0.95,
)
STATE_PREPARATION = "reference"  # Set to "product" for a logical-zero smoke test.
HARDWARE_STATE = "two_qutrit"
HARDWARE_ENCODING = "canonical"
IBM_DEVICE = "ibm_your_device"
IQM_DEVICE = "your_iqm_device"
RUN_IBM = False
RUN_IQM = False
RUN_PIASTQ = False
RESULTS = {}


The helper creates fresh inputs under `artifacts/bell_randomized_raw/inputs`. It prepares the audited graph state using local encoded `|+>` states and its weighted edges. The optional product mode prepares logical zero on every party. These demonstration circuits are portable inputs, not optimized hardware candidates; existing optimized directories can be supplied with `PathBasis`.

The encodings below are computational-basis embeddings, which lets the helper build graph phases directly. Measurement rotations and outcome decoding are handled by the experiment pipeline, including the changed leakage input word in the permuted encoding.


In [ ]:
def prepare_basis(state, encoding_name):
    reference = get_reference_experiment(state)
    e = ENCODINGS[encoding_name]
    parties = reference.state.num_parties
    pairs = tuple((2 * i, 2 * i + 1) for i in reversed(range(parties)))
    circuit = QuantumCircuit(2 * parties)
    if STATE_PREPARATION not in {"reference", "product"}:
        raise ValueError("STATE_PREPARATION must be reference or product")
    logical_local = (
        np.ones(3, dtype=complex) / np.sqrt(3)
        if STATE_PREPARATION == "reference"
        else np.array([1, 0, 0], dtype=complex)
    )
    for pair in pairs:
        circuit.append(StatePreparation(e @ logical_local), pair)
    if STATE_PREPARATION == "reference":
        physical_levels = np.argmax(np.abs(e), axis=0)
        root = np.exp(2j * np.pi / 3)
        for first, second, weight in reference.state.weighted_edges:
            diagonal = np.ones(16, dtype=complex)
            for a in range(3):
                for b in range(3):
                    diagonal[4 * physical_levels[a] + physical_levels[b]] = root ** ((weight * a * b) % 3)
            circuit.append(DiagonalGate(diagonal), (*pairs[second], *pairs[first]))
    directory = WORK_ROOT / "inputs" / f"{state}-{encoding_name}-{uuid4().hex}"
    directory.mkdir(parents=True)
    with (directory / "graph_state_direct_basis.qpy").open("wb") as handle:
        qpy.dump(circuit, handle)
    np.save(directory / "E.npy", e)
    return directory


The local comparison below runs six separate experiments: three scenarios times two encodings. The lower bound `N_min = setting_draws` is followed by additional independent draws until every required pattern is covered. If `N_max = max_setting_draws` is reached first, the run records `coverage_limit_reached` and submits nothing. Actual raw cost is `N_actual × K`, not necessarily `N_min × K`.

There are 9/18/36 full configurations and 9/12/13 required patterns for the three scenarios. A `None` in an AME pattern means an identity: the party still receives a physical setting and is measured. Its outcome is marginalized, while its leakage still invalidates the joint shot. GHZ has 6/18 and AME 21/36 configurations that contribute to no Bell term. These zero-contribution blocks remain in the schedule, raw data, shot cost, and global leakage statistics.


In [ ]:
for state in STATES:
    for encoding_name in ENCODINGS:
        basis_directory = prepare_basis(state, encoding_name)
        specification = ExperimentSpec(
            state=state,
            basis=PathBasis(basis_directory),
            backend=AerIdeal(seed_simulator=17),
            measurement=MEASUREMENT,
            output_root=WORK_ROOT / "runs",
            tags={"example": "randomized_raw", "encoding": encoding_name},
        )
        RESULTS[("aer", state, encoding_name)] = run_experiment(
            specification, repo_root=REPO_ROOT,
        )


Hardware execution is opt-in. Configure the provider account outside this notebook, replace the device placeholders, choose the state/encoding and budget, then enable only the intended flag. IBM uses Runtime SamplerV2 with dynamical decoupling and both twirling options disabled. IQM uses its native compilation; PIAST/AQT uses the existing managed runner and default local transpilation configuration.

Raw v1 rejects readout mitigation, circuit twirling, ZNE, forced recalibration, and incompatible execution overrides. Do not pass legacy `shots`, `uncertainty`, or `bootstrap` together with `RandomizedBlocks`; use `measurement.shots_per_draw` and `measurement.confidence_level`.


In [ ]:
if RUN_IBM:
    ibm_specification = ExperimentSpec(
        state=HARDWARE_STATE,
        basis=PathBasis(prepare_basis(HARDWARE_STATE, HARDWARE_ENCODING)),
        backend=IBMHardware(device=IBM_DEVICE),
        measurement=MEASUREMENT,
        output_root=WORK_ROOT / "runs",
        tags={"example": "randomized_raw", "encoding": HARDWARE_ENCODING},
    )
    RESULTS[("ibm", HARDWARE_STATE, HARDWARE_ENCODING)] = run_experiment(
        ibm_specification, repo_root=REPO_ROOT,
    )


In [ ]:
if RUN_IQM:
    iqm_specification = ExperimentSpec(
        state=HARDWARE_STATE,
        basis=PathBasis(prepare_basis(HARDWARE_STATE, HARDWARE_ENCODING)),
        backend=IQMHardware(device=IQM_DEVICE),
        measurement=MEASUREMENT,
        output_root=WORK_ROOT / "runs",
        tags={"example": "randomized_raw", "encoding": HARDWARE_ENCODING},
    )
    RESULTS[("iqm", HARDWARE_STATE, HARDWARE_ENCODING)] = run_experiment(
        iqm_specification, repo_root=REPO_ROOT,
    )


In [ ]:
if RUN_PIASTQ:
    piastq_specification = ExperimentSpec(
        state=HARDWARE_STATE,
        basis=PathBasis(prepare_basis(HARDWARE_STATE, HARDWARE_ENCODING)),
        backend=PiastQHardware(mode="managed"),
        measurement=MEASUREMENT,
        output_root=WORK_ROOT / "runs",
        tags={"example": "randomized_raw", "encoding": HARDWARE_ENCODING},
    )
    RESULTS[("piastq", HARDWARE_STATE, HARDWARE_ENCODING)] = run_experiment(
        piastq_specification, repo_root=REPO_ROOT,
    )


The main `raw` Bell value assigns zero contribution to a joint shot with leakage at any party. `conditional` separately normalizes each correlator by its matched accepted shots. It is not the raw value divided by global acceptance. If a required correlator has no accepted shots, `conditional` is null with `no_accepted_shots`; complete raw acquisition can still have completed execution status.

The method `conditional_schedule_block_hoeffding_v1` gives conservative intervals conditional on the saved schedule. It assumes independent blocks and allows arbitrary dependence among shots within one block. Increasing K does not replace acquiring more independent blocks. Single-block and constant observations may still have wide intervals. The result includes AME diagnostics per measured full context; these contexts are not selected after inspecting their values.

Classical bounds are diagnostic comparisons requiring faithful local measurements and, for a stationary interpretation, stable measurements. Postselection introduces additional assumptions. This workflow does not establish a loophole-closing Bell test or a nonlocality p-value.


In [ ]:
def summarize_result(key, result):
    backend, state, encoding_name = key
    values = result.values
    schedule = json.loads((result.artifact_dir / "schedule.json").read_text(encoding="utf-8"))
    budget = values.get("budget", {})
    uncertainty = values.get("uncertainty", {})
    reference = get_reference_experiment(state)
    return {
        "backend": backend,
        "state": state,
        "encoding": encoding_name,
        "status": result.status.value,
        "N_min": schedule["config"]["setting_draws"],
        "N_max": schedule["config"]["max_setting_draws"],
        "N_actual": len(schedule["blocks"]),
        "K": schedule["config"]["shots_per_draw"],
        "scheduled_shots": len(schedule["blocks"]) * schedule["config"]["shots_per_draw"],
        "completed_shots": budget.get("completed_shots", 0),
        "coverage": values.get("coverage"),
        "missing_scheduled_patterns": schedule["missing_pattern_indices"],
        "zero_contribution_blocks": sum(not block["matched_pattern_indices"] for block in schedule["blocks"]),
        "raw": values.get("raw"),
        "conditional": values.get("conditional"),
        "conditional_reason": values.get("conditional_reason"),
        "leakage": values.get("leakage", {}).get("global"),
        "interval_method": uncertainty.get("method"),
        "raw_interval": uncertainty.get("raw"),
        "conditional_interval": uncertainty.get("conditional"),
        "quality_flags": values.get("quality_flags", []),
        "diagnostic_classical_bound": reference.bell_functional.classical_bound,
        "artifact_directory": str(result.artifact_dir),
    }

SUMMARY = [summarize_result(key, result) for key, result in RESULTS.items()]
print(json.dumps(SUMMARY, indent=2, allow_nan=False))


Every run records its configuration, reference, full schedule, circuit catalogue, ordered batch requests, returned job identifiers, raw counts, and derived analysis. Repeated blocks remain distinct even when they reuse a compiled circuit.

The next cell reads the saved report and calls the public resume API on an already completed local run. Completed runs need no backend connection. For an interrupted run, supply its saved directory to the same API: confirmed jobs are retrieved and untouched batches can continue. An ambiguous submission is marked `submission_unknown`; a missing job ID is not permission to submit again. Raw data survives failed analysis and can be reanalyzed locally. A new intentional measurement should create a new run.


In [ ]:
completed = next(
    (result for result in RESULTS.values() if result.status.value == "completed"),
    None,
)
RESTORED = None
if completed is not None:
    SAVED_DIRECTORY = completed.artifact_dir
    SAVED_REPORT = json.loads((SAVED_DIRECTORY / "analysis" / "bell.json").read_text(encoding="utf-8"))
    RESTORED = resume_experiment(SAVED_DIRECTORY)
    print("Saved raw Bell value:", SAVED_REPORT["raw"])
    print("Reloaded status:", RESTORED.status.value)
